# 02 Text Preprocessing & Modern Tokenization Paradigms

**Real-World Scenario**: Enterprise Procurement Contract Tokenization & Clause Parsing System.

This notebook demonstrates comparing Stemming vs. Lemmatization, executing Byte Pair Encoding (BPE) subword tokenization with `tiktoken`, saving artifacts into a hidden `.model_cache/` directory, and executing a **Complete GenAI Procurement Contract Extraction LLM Call**.

## Step 1: Environment Setup & Local Cache Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

# Create hidden model cache directory for local pipeline artifact persistence
os.makedirs(".model_cache", exist_ok=True)

print("=== Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden Cache Directory Path:", os.path.abspath(".model_cache"))

## Step 2: Enterprise Procurement Contracts Dataset Setup

In [ ]:
contracts = [
    "Clause 14.2: Procurement payments of $150,000 USD for cloud infrastructure services rendered in Q1 2026 must be settled within 30 business days.",
    "Clause 18.5: Unexplainable system downtime exceeding 0.05% SLA triggers automatic financial penalty refunds to Acme Enterprise Corp.",
    "Clause 22.1: Non-disclosure obligations remain strictly enforceable for 7 years following contract termination."
]

print(f"Loaded {len(contracts)} enterprise procurement contract clauses.")
print("Sample Clause #1:", contracts[0])

## Step 3: Comparing Stemming vs. Lemmatization vs. Subword BPE Tokenization

In [ ]:
import tiktoken
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
enc = tiktoken.get_encoding("cl100k_base") # OpenAI BPE Tokenizer

sample_words = ["enforceable", "payments", "unexplainable", "settled"]

print("=== Stemming vs Subword BPE Comparison ===")
print(f"{'Original Word':<15} | {'Porter Stemmer':<15} | {'BPE Subwords (cl100k_base)':<30}")
print("-" * 65)

for w in sample_words:
    stem_val = stemmer.stem(w)
    token_ids = enc.encode(w)
    subwords = [enc.decode([t]) for t in token_ids]
    print(f"{w:<15} | {stem_val:<15} | {str(subwords):<30}")

## Step 4: Tokenizing Full Contract Clause Dataset with OpenAI BPE Tokenizer

In [ ]:
tokenized_contracts = []
for idx, clause in enumerate(contracts, 1):
    t_ids = enc.encode(clause)
    subwords = [enc.decode([t]) for t in t_ids]
    tokenized_contracts.append({"clause_id": idx, "tokens": t_ids, "subwords": subwords})
    print(f"Clause #{idx} Token Length: {len(t_ids)} tokens")
    print(f"  First 5 Subwords: {subwords[:5]}")

## Step 5: Executing Complete GenAI Procurement Contract Extraction LLM Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if os.getenv("OPENAI_API_KEY"):
    prompt = ChatPromptTemplate.from_template('''You are an Enterprise Legal Procurement AI Assistant.
Analyze the procurement contract clauses below and extract structured metadata.

Contract Clauses:
{clauses}

Extraction Requirements:
1. Total Payment Amount & Settlement Period.
2. SLA Downtime Threshold & Penalty Trigger.
3. Non-Disclosure Enforceability Period.

Extraction Summary:''')
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    formatted_clauses = "\n".join([f"- {c}" for c in contracts])
    
    response = llm.invoke(prompt.format(clauses=formatted_clauses))
    print("=== Complete GenAI Procurement Extraction Response ===")
    print(response.content)
else:
    print("[NOTICE] OPENAI_API_KEY not found in .env. Skipping live LLM call.")